# SmartVision AI — Phase 2: Transfer Learning (25-class classification)

Train **VGG16, ResNet50, MobileNetV2, EfficientNetB0** on the cropped 224×224 subset.

**Runtime:** Google Colab **T4 GPU**. Enable: Runtime → Change runtime type → T4 GPU.

**Input:** `smartvision_dataset/classification/{train,val,test}/<class>/*.jpg` from `Smartvision.ipynb`.

**Output:**
- `models/vgg16.keras`, `resnet50.keras`, `mobilenetv2.keras`, `efficientnetb0.keras`
- `reports/classification_metrics.json`
- confusion matrices under `reports/figures/`

Rubric floor: **test accuracy ≥ 80%**. If a model lands under that, the recovery cell unfreezes more layers using *that run's* val curve.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install tensorflow pandas scikit-learn matplotlib seaborn pillow tqdm

In [ ]:
import os, sys, json, time, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, optimizers, callbacks

print("TF", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    if zip_path.exists() and not (data_dir / "classification" / "train").exists():
        import zipfile
        print("Unzipping dataset...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_dir if not any(data_dir.glob("*")) else PROJECT_ROOT)
        # handle zip that contains smartvision_dataset/ as root or as folder
        if not (data_dir / "classification").exists():
            inner = PROJECT_ROOT / "smartvision_dataset"
            print("classification path exists:", (inner / "classification").exists())
    DRIVE_REPO = Path("/content/drive/MyDrive/Smart_Vision_AI")
    if DRIVE_REPO.exists():
        sys.path.insert(0, str(DRIVE_REPO))
    sys.path.insert(0, str(PROJECT_ROOT))
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    OUT_ROOT = PROJECT_ROOT

CLASS_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "truck",
    "traffic light", "stop sign", "bench", "bird", "cat", "dog", "horse",
    "cow", "elephant", "bottle", "cup", "bowl", "pizza", "cake", "chair",
    "couch", "potted plant", "bed",
]
NUM_CLASSES = 25
IMAGE_SIZE = 224
BATCH = 16  # 16 is safer on Colab T4 with VGG16 flatten; raise to 32 if memory allows

DATA = PROJECT_ROOT / "smartvision_dataset" / "classification"
# fallback if unzip extracted into cwd
if not DATA.exists():
    alt = Path("/content/smartvision_dataset/classification")
    if alt.exists():
        DATA = alt
print("DATA =", DATA, "exists", DATA.exists())

MODELS_DIR = OUT_ROOT / "models"
FIGURES = OUT_ROOT / "reports" / "figures"
REPORTS = OUT_ROOT / "reports"
for p in (MODELS_DIR, FIGURES, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

# Count files actually on disk — do not assume 70/15/15 landed exactly
def count_split(split):
    rows = {}
    root = DATA / split
    for n in CLASS_NAMES:
        folder = root / n
        rows[n] = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    return rows

for split in ("train", "val", "test"):
    c = count_split(split)
    print(f"{split:5s} total={sum(c.values()):4d}  min_class={min(c.values())} max_class={max(c.values())} missing={[k for k,v in c.items() if v==0]}")
assert sum(count_split("train").values()) > 0, "No training images. Run Smartvision.ipynb first / unzip Drive zip." 

In [ ]:
## tf.data pipelines + the brief's augmentation set
# flip, rotation ±15°, brightness ±20%, contrast, zoom, color jitter (saturation/hue)

def list_image_label(split):
    paths, labels = [], []
    root = DATA / split
    for idx, name in enumerate(CLASS_NAMES):
        for f in sorted((root / name).glob("*.jpg")):
            paths.append(str(f))
            labels.append(idx)
    return tf.constant(paths), tf.constant(labels, dtype=tf.int32)

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = tf.cast(img, tf.float32)  # 0..255 — model-specific preprocess_input handles scaling
    return img, label

# Keras preprocessing layers: rotation of 15/360 ≈ 0.0417 of a full turn
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15.0 / 360.0, fill_mode="nearest"),
    layers.RandomBrightness(0.20),
    layers.RandomContrast(0.20),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.05, 0.05),
], name="brief_augmentation")

def color_jitter(img, label):
    img = tf.image.random_saturation(img / 255.0, 0.7, 1.3) * 255.0
    img = tf.image.random_hue(img / 255.0, 0.05) * 255.0
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_ds(split, training=False, batch=BATCH, extra_advanced=False):
    paths, labels = list_image_label(split)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(color_jitter, num_parallel_calls=tf.data.AUTOTUNE)
        if extra_advanced:
            # MixUp (bonus advanced augmentation) — applied later in EfficientNet path
            pass
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds, len(paths)

train_ds, n_train = make_ds("train", training=True)
val_ds, n_val = make_ds("val", training=False)
test_ds, n_test = make_ds("test", training=False)
print(f"n_train={n_train} n_val={n_val} n_test={n_test}")

# Peek one augmented batch so we know augmentation is actually firing
xb, yb = next(iter(train_ds))
print("batch", xb.shape, xb.dtype, "range", float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(tf.cast(tf.clip_by_value(xb[i], 0, 255), tf.uint8))
    ax.set_title(CLASS_NAMES[int(yb[i])], fontsize=8)
    ax.axis("off")
fig.suptitle("Augmented training crops (brief transforms)")
fig.tight_layout()
fig.savefig(FIGURES / "aug_preview.png", dpi=130)
plt.show()

In [ ]:
def mixup_ds(ds, alpha=0.2):
    # Advanced MixUp for EfficientNet only (bonus +2)
    def _mix(batch_x, batch_y):
        lam = tf.random.uniform([], minval=0.0, maxval=1.0)
        # Beta(alpha, alpha) via two gammas
        g1 = tf.random.gamma([], alpha)
        g2 = tf.random.gamma([], alpha)
        lam = g1 / (g1 + g2)
        idx = tf.random.shuffle(tf.range(tf.shape(batch_x)[0]))
        mixed_x = lam * batch_x + (1.0 - lam) * tf.gather(batch_x, idx)
        y1 = tf.one_hot(batch_y, NUM_CLASSES)
        y2 = tf.one_hot(tf.gather(batch_y, idx), NUM_CLASSES)
        mixed_y = lam * y1 + (1.0 - lam) * y2
        return mixed_x, mixed_y
    return ds.map(_mix, num_parallel_calls=tf.data.AUTOTUNE)

def common_callbacks(name, patience=6):
    return [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=patience, restore_best_weights=True, mode="max"),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint(str(MODELS_DIR / f"{name}.keras"), monitor="val_accuracy", save_best_only=True, mode="max"),
        callbacks.CSVLogger(str(REPORTS / f"{name}_history.csv")),
    ]

def plot_history(hist, name):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(hist.history["accuracy"], label="train")
    ax[0].plot(hist.history["val_accuracy"], label="val")
    ax[0].set_title(f"{name} accuracy"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(hist.history["loss"], label="train")
    ax[1].plot(hist.history["val_loss"], label="val")
    ax[1].set_title(f"{name} loss"); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}_history.png", dpi=130)
    plt.show()

def evaluate_model(model, name):
    y_true, y_prob = [], []
    t0 = time.perf_counter()
    n = 0
    for xb, yb in test_ds:
        p = model.predict(xb, verbose=0)
        y_true.append(yb.numpy())
        y_prob.append(p)
        n += xb.shape[0]
    elapsed = time.perf_counter() - t0
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = y_prob.argmax(axis=1)
    top5 = np.mean([yt in np.argsort(pr)[-5:] for yt, pr in zip(y_true, y_prob)])
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    prec_c, rec_c, f1_c, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(11, 10))
    sns.heatmap(cm, cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{name} confusion matrix (test)")
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIGURES / f"cm_{name}.png", dpi=140)
    plt.show()
    size_mb = (MODELS_DIR / f"{name}.keras").stat().st_size / 1e6 if (MODELS_DIR / f"{name}.keras").exists() else 0.0
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
    print(report)
    per_class = {
        CLASS_NAMES[i]: {"precision": float(prec_c[i]), "recall": float(rec_c[i]), "f1": float(f1_c[i]), "support": int(sup[i])}
        for i in range(NUM_CLASSES)
    }
    metrics = {
        "model": name,
        "accuracy": acc,
        "precision_macro": float(prec),
        "recall_macro": float(rec),
        "f1_macro": float(f1),
        "top5_accuracy": float(top5),
        "inference_ms": float(elapsed / max(n, 1) * 1000),
        "model_size_mb": float(size_mb),
        "n_test": int(n),
        "per_class": per_class,
    }
    print(name, {k: metrics[k] for k in ("accuracy", "f1_macro", "top5_accuracy", "inference_ms", "model_size_mb")})
    return metrics, cm

ALL_METRICS = {}

In [ ]:
## Model 1 — VGG16 (frozen conv base, dropout head)

base = applications.VGG16(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
base.trainable = False
inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = applications.vgg16.preprocess_input(inp)
x = base(x, training=False)
x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
vgg = models.Model(inp, out, name="VGG16")
vgg.compile(optimizer=optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy",
            metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
vgg.summary()
hist_vgg = vgg.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=common_callbacks("vgg16"), verbose=1)
plot_history(hist_vgg, "vgg16")
vgg = keras.models.load_model(MODELS_DIR / "vgg16.keras")
ALL_METRICS["VGG16"], _ = evaluate_model(vgg, "vgg16")

In [ ]:
## Model 2 — ResNet50 (unfreeze last 20 layers, GAP head, LR schedule via ReduceLROnPlateau)

base = applications.ResNet50(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
for layer in base.layers:
    layer.trainable = False
for layer in base.layers[-20:]:
    layer.trainable = True
print("Trainable layers:", sum(1 for l in base.layers if l.trainable), "/", len(base.layers))

inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = applications.resnet50.preprocess_input(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
resnet = models.Model(inp, out, name="ResNet50")
resnet.compile(optimizer=optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy",
               metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
hist_rn = resnet.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=common_callbacks("resnet50"), verbose=1)
plot_history(hist_rn, "resnet50")
resnet = keras.models.load_model(MODELS_DIR / "resnet50.keras")
ALL_METRICS["ResNet50"], _ = evaluate_model(resnet, "resnet50")

In [ ]:
## Model 3 — MobileNetV2 (frozen base, compact head, speed-focused)

base = applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
base.trainable = False
inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = applications.mobilenet_v2.preprocess_input(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
mnet = models.Model(inp, out, name="MobileNetV2")
mnet.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy",
             metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
hist_mn = mnet.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=common_callbacks("mobilenetv2"), verbose=1)
plot_history(hist_mn, "mobilenetv2")
mnet = keras.models.load_model(MODELS_DIR / "mobilenetv2.keras")
ALL_METRICS["MobileNetV2"], _ = evaluate_model(mnet, "mobilenetv2")

In [ ]:
## Model 4 — EfficientNetB0 (mixed precision, BN head, MixUp advanced aug)

keras.mixed_precision.set_global_policy("mixed_float16")
base = applications.EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
base.trainable = False
inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = applications.efficientnet.preprocess_input(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)  # float32 softmax under mixed precision
eff = models.Model(inp, out, name="EfficientNetB0")
eff.compile(optimizer=optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy",
            metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])

train_mix = mixup_ds(train_ds)
# MixUp yields soft labels — compile with categorical loss for this stage
eff.compile(optimizer=optimizers.Adam(1e-4), loss="categorical_crossentropy",
            metrics=["accuracy"])
hist_eff1 = eff.fit(train_mix, validation_data=val_ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES))),
                    epochs=12, callbacks=common_callbacks("efficientnetb0", patience=5), verbose=1)

# Stage 2: unfreeze last 20 backbone layers, sparse labels, lower LR
keras.mixed_precision.set_global_policy("mixed_float16")
eff = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")
# find backbone
backbone = None
for layer in eff.layers:
    if "efficientnet" in layer.name:
        backbone = layer
        break
if backbone is not None:
    backbone.trainable = True
    for layer in backbone.layers[:-20]:
        layer.trainable = False
    print("EfficientNet trainable layers", sum(1 for l in backbone.layers if l.trainable), "/", len(backbone.layers))

eff.compile(optimizer=optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy",
            metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
hist_eff2 = eff.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("efficientnetb0", patience=4), verbose=1)
plot_history(hist_eff2, "efficientnetb0")
keras.mixed_precision.set_global_policy("float32")
eff = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")
ALL_METRICS["EfficientNetB0"], _ = evaluate_model(eff, "efficientnetb0")

In [ ]:
## Recovery: any model under 80% test accuracy — unfreeze more and keep training
# Decision is based on THIS run's metrics, not a preset.

def deepen_finetune(keras_path, preprocess_fn, base_name_substr, n_unfreeze, lr, out_name, epochs=8):
    model = keras.models.load_model(keras_path)
    backbone = None
    for layer in model.layers:
        if base_name_substr in layer.name:
            backbone = layer
            break
    if backbone is None:
        print("No backbone named", base_name_substr)
        return model
    backbone.trainable = True
    keep_frozen = max(0, len(backbone.layers) - n_unfreeze)
    for layer in backbone.layers[:keep_frozen]:
        layer.trainable = False
    print(out_name, "trainable", sum(1 for l in backbone.layers if l.trainable), "/", len(backbone.layers))
    model.compile(optimizer=optimizers.Adam(lr), loss="sparse_categorical_crossentropy",
                  metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=common_callbacks(out_name, patience=3), verbose=1)
    return keras.models.load_model(MODELS_DIR / f"{out_name}.keras")

RECOVERY = {
    "VGG16": ("vgg16", "vgg16", 8, 1e-5),
    "ResNet50": ("resnet50", "resnet50", 40, 1e-5),
    "MobileNetV2": ("mobilenetv2", "mobilenetv2", 40, 1e-5),
    "EfficientNetB0": ("efficientnetb0", "efficientnet", 40, 5e-6),
}
for display_name, (file_stem, substr, n_unfreeze, lr) in RECOVERY.items():
    acc = ALL_METRICS[display_name]["accuracy"]
    print(f"{display_name} test acc={acc:.3f}")
    if acc < 0.80:
        print(f"  -> below 80%. Unfreezing last {n_unfreeze} backbone layers (lr={lr}) based on this run.")
        m = deepen_finetune(MODELS_DIR / f"{file_stem}.keras", None, substr, n_unfreeze, lr, file_stem, epochs=10)
        ALL_METRICS[display_name], _ = evaluate_model(m, file_stem)
    else:
        print("  -> meets 80% floor; no extra unfreeze.")

In [ ]:
## Persist classification metrics

payload = {
    "class_names": CLASS_NAMES,
    "n_train": int(n_train),
    "n_val": int(n_val),
    "n_test": int(n_test),
    "classification": ALL_METRICS,
    "best_classification_model": max(ALL_METRICS, key=lambda k: ALL_METRICS[k]["accuracy"]),
}
(REPORTS / "classification_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "per_class"} for k, v in ALL_METRICS.items()}, indent=2))
print("Best:", payload["best_classification_model"])
print("Saved models:", list(MODELS_DIR.glob("*.keras")))
print("Copy these to the GitHub repo models/ folder after download.")